# PYNQ-Z2 ANC Project - Setup and FPGA Programming

This notebook will:
1. Program the FPGA with the ANC bitstream
2. Verify basic I/O functionality (LEDs, switches, buttons)
3. Confirm the design is loaded and operational

## Prerequisites
- Ensure the bitstream file `anc_project.bit` is in the same directory or accessible
- Ensure the FPGA overlay is built for PYNQ-Z2

## 1. Import PYNQ Libraries

In [ ]:
from pynq import Overlay
from pynq import MMIO
import time

print("PYNQ libraries imported successfully")

## 2. Load the FPGA Bitstream

Download the ANC design bitstream to the FPGA programmable logic.

In [ ]:
# Load the bitstream
# Note: Replace 'anc_project.bit' with your actual bitstream filename
bitstream_path = 'anc_project.bit'

try:
    overlay = Overlay(bitstream_path)
    print(f"Bitstream loaded successfully from {bitstream_path}")
    print(f"IP dictionary: {overlay.ip_dict}")
except Exception as e:
    print(f"Error loading bitstream: {e}")
    print("\nNote: Ensure the bitstream file exists and the design is correctly built.")

## 3. Access GPIO and Control Registers

The design exposes the following I/O:
- `btn_rst_n`: Reset button (active-low)
- `sw_bypass`: Bypass switch (0 = ANC active, 1 = Raw bypass)
- `leds[3:0]`: Convergence status LEDs

These are typically mapped to AXI GPIO controllers.

In [ ]:
# Try to find GPIO controllers in the overlay
gpio_controllers = {}

for ip_name, ip_info in overlay.ip_dict.items():
    if 'gpio' in ip_name.lower():
        gpio_controllers[ip_name] = ip_info
        print(f"Found GPIO controller: {ip_name}")
        print(f"  State: {ip_info['state']}")
        print(f"  Registers: {ip_info['registers']}")

if not gpio_controllers:
    print("No GPIO controllers found in the overlay.")
    print("The design may use direct I/O or different IP blocks.")

## 4. Direct Memory-Mapped I/O Access

If the design uses a custom IP block with AXI-Lite interface, we can access it directly via MMIO.

In [ ]:
# Find the top-level ANC controller IP
anc_controller = None

for ip_name, ip_info in overlay.ip_dict.items():
    if 'anc' in ip_name.lower() or 'top' in ip_name.lower():
        anc_controller = ip_name
        print(f"Found ANC controller: {ip_name}")
        print(f"  Address: 0x{ip_info['phys_addr']:08X}")
        print(f"  Size: {ip_info['addr_range']} bytes")
        break

if anc_controller:
    # Create MMIO object for the controller
    mmio = MMIO(overlay.ip_dict[anc_controller]['phys_addr'], 
               overlay.ip_dict[anc_controller]['addr_range'])
    print(f"\nMMIO object created for {anc_controller}")
else:
    print("No ANC controller IP found. Check the Vivado design.")

## 5. Test LED Control

Test that we can read/write to the LED outputs to verify the datapath is active.

In [ ]:
if anc_controller:
    # Try to read LED status (offset depends on your design)
    # This is a placeholder - adjust offset based on your AXI-Lite register map
    LED_OFFSET = 0x00  # Example offset - adjust based on your design
    
    try:
        led_value = mmio.read(LED_OFFSET)
        print(f"LED register value: 0x{led_value:04X} (binary: {led_value:04b})")
        print(f"\nLED states:")
        for i in range(4):
            print(f"  LED{i}: {'ON' if (led_value >> i) & 1 else 'OFF'}")
    except Exception as e:
        print(f"Error reading LED register: {e}")
        print("\nThe LED offset may need adjustment based on your design.")
else:
    print("Skipping LED test - no controller found.")

## 6. Test Switch Input

Read the bypass switch (SW0) to verify input path.

In [ ]:
if anc_controller:
    # Try to read switch status
    SWITCH_OFFSET = 0x04  # Example offset - adjust based on your design
    
    try:
        switch_value = mmio.read(SWITCH_OFFSET)
        print(f"Switch register value: 0x{switch_value:04X}")
        print(f"SW0 (Bypass): {'BYPASS MODE' if switch_value & 1 else 'ANC MODE'}")
    except Exception as e:
        print(f"Error reading switch register: {e}")
else:
    print("Skipping switch test - no controller found.")

## 7. Monitor LED Status Over Time

Watch the LEDs to see if the step-size controller is active (even without audio input).

In [ ]:
if anc_controller:
    print("Monitoring LED status for 10 seconds...")
    print("(Observe onboard LEDs LD0-LD3)")
    
    for i in range(10):
        try:
            led_value = mmio.read(LED_OFFSET)
            switch_value = mmio.read(SWITCH_OFFSET)
            print(f"[{i}s] LEDs: {led_value:04b} | SW0: {switch_value & 1}")
        except:
            print(f"[{i}s] Unable to read registers")
        time.sleep(1)
    
    print("\nMonitoring complete.")
else:
    print("Skipping monitoring - no controller found.")

## 8. Manual I/O Test

Instructions for manual verification:
1. Press BTN0 (reset) - LEDs should reset
2. Toggle SW0 - should switch between ANC and bypass modes
3. Observe LD0-LD3 - these indicate step-size convergence level

In [ ]:
print("Manual I/O Test Instructions:")
print("="*50)
print("1. Press BTN0 (reset button)")
print("   - Observe: All LEDs should turn off or reset to initial state")
print("\n2. Toggle SW0 (bypass switch)")
print("   - Position 0: ANC active mode")
print("   - Position 1: Bypass mode (raw audio)")
print("\n3. Observe LD0-LD3")
print("   - These show the active step-size shift value")
print("   - More LEDs lit = finer step-size (better convergence)")
print("\n" + "="*50)

## 9. Check Clock Domains

Verify that the clock is running at the expected frequency (125 MHz system clock).

In [ ]:
# Check PS (Processing System) clock
from pynq.ps import Clocks

clocks = Clocks()
print("Processing System Clocks:")
print(f"  PL clock: {clocks.get_pl_clk() / 1e6:.2f} MHz")
print(f"  CPU clock: {clocks.get_cpu_clk() / 1e6:.2f} MHz")

## 10. Summary

### Verification Checklist:
- [ ] Bitstream loaded successfully
- [ ] GPIO/controllers accessible
- [ ] LED read/write functional
- [ ] Switch input readable
- [ ] Manual button/switch test passed
- [ ] Clock domains verified

### Next Steps:
- Proceed to `02_audio_codec_test.ipynb` to test the I2S codec interface
- Ensure audio codec is connected to Pmod JA
- Verify audio input/output path